In [ ]:
# ── PIPELINE REQUIREMENTS ─────────────────────────────────────────────
# Run these in your terminal environment using your active uv venv, or uncomment below:
# !uv pip install vllm nest_asyncio

SAVE_EVAL = False   # Set to False when running on the private test set


import os
import json
import nest_asyncio
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import re
import sys

# Apply notebook hotfix for AsyncLLMEngine compatibility
nest_asyncio.apply()

# Accelerate model weight downloads using parallel file transfer threads
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# ── GLOBAL RUNTIME CONFIGURATION ──────────────────────────────────────
# MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507" 
DATA_PATH = "data/public.jsonl"

if SAVE_EVAL == False:
    DATA_PATH = "data/private.jsonl"
OUTPUT_PATH = "results/vllm_predictions.jsonl"

# VRAM budget management properties
# MAX_MODEL_LEN = 32768  
# MAX_TOKENS_TO_GENERATE = 16384  

# Change the model path to the officially supported AWQ variant
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507" 

# Tighten the max context window to protect laptop VRAM from spikes
MAX_MODEL_LEN = 16384  
MAX_TOKENS_TO_GENERATE = 4096

SAMPLE_LIMIT = 5
SAMPLE_IDX = 0

HF_TOKEN = ""
os.environ["HF_TOKEN"] = HF_TOKEN

print("Notebook runtime environment configured successfully.")

Notebook runtime environment configured successfully.


In [2]:
from vllm import LLM, SamplingParams

# ── MATHEMATICAL CONSTRAINTS & FORMATTING PROMPTS ─────────────────────
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

# ── DATA PROCESSING FUNCTIONS ─────────────────────────────────────────
def load_and_prepare_prompts(file_path: str, llm_engine: LLM) -> Tuple[List[str], List[Dict[str, Any]]]:
    tokenizer = llm_engine.get_tokenizer()
    formatted_prompts = []
    metadata = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            
            question = item["question"]
            options = item.get("options", None)
            is_mcq = bool(options)
            
            if is_mcq:
                labels = [chr(65 + i) for i in range(len(options))]
                opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
                user_content = f"{question}\n\nOptions:\n{opts_text}"
                system_content = SYSTEM_PROMPT_MCQ
            else:
                user_content = question
                system_content = SYSTEM_PROMPT_MATH
                
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": user_content}
            ]
            
            # Formats with the native Qwen3 chat template block 
            full_prompt_string = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            
            formatted_prompts.append(full_prompt_string)
            metadata.append({
                "id": item["id"],
                "is_mcq": is_mcq,
                "gold": item.get("gold", None)
            })
            
    return formatted_prompts, metadata

print("Data processing engines compiled.")

Data processing engines compiled.


In [ ]:
import torch
print(f"Spawning local background vLLM instance for: {MODEL_NAME}...")

# Update the model string to the 4B thinking architecture
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"
MAX_MODEL_LEN = 4096

print(f"Spawning background vLLM instance for: {MODEL_NAME}...")
print("Applying on-the-fly INT8 quantization via BitsAndBytes...")

llm = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_MODEL_LEN,
    tensor_parallel_size=1,
    
    # ── ON-THE-FLY INT8 COMPRESSION ──
    quantization="bitsandbytes",  # Forces vLLM to compress weights to 8-bit integers
    dtype=torch.bfloat16,        # Keeps internal math stable while parsing
    
    # ── MEMORY SAFETY BUFFER ──
    gpu_memory_utilization=0.85, # Leaves 15% VRAM free for the system and KV Cache
    enforce_eager=True,          # Skips heavy CUDA graph captures to save VRAM
    trust_remote_code=True,
    
    # ── AUTHENTICATION & REASONING ──
    hf_token=HF_TOKEN,
    structured_outputs_config={
        "reasoning_parser": "qwen3"
    }
)


In [4]:
print(f"Loading tracking instances from {DATA_PATH}...")
all_prompts, all_metadata = load_and_prepare_prompts(DATA_PATH, llm)

# ── THE SLICING MODIFICATION ──────────────────────────────────────────
# If SAMPLE_LIMIT is set, grab only the first 'n' elements using Python list slicing
if 'SAMPLE_LIMIT' in globals() and SAMPLE_LIMIT is not None:
    start = SAMPLE_IDX*SAMPLE_LIMIT
    end = (SAMPLE_IDX+1)*SAMPLE_LIMIT
    prompts = all_prompts[start:end]
    metadata = all_metadata[start:end]
    print(f"⚠️ SMOKE TEST ENABLED: Truncated dataset to the first {len(prompts)} items.")
else:
    prompts = all_prompts
    metadata = all_metadata
    print(f"Full run enabled. Staging all {len(prompts)} prompts for inference.")

Loading tracking instances from data/private.jsonl...
⚠️ SMOKE TEST ENABLED: Truncated dataset to the first 5 items.


In [5]:
# Official recommended configuration bounds for Qwen3-Thinking variants:
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    presence_penalty=1.2,  # Strong tracking defense protecting against recursive loops
    max_tokens=MAX_TOKENS_TO_GENERATE,
    stop=["<|im_end|>", "<|endoftext|>"]
)

print(f"Spinning engine batches across {len(prompts)} records inside Jupyter loop...")
outputs = llm.generate(prompts, sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]
print("Inference generation processing complete.")

Spinning engine batches across 5 records inside Jupyter loop...


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

(EngineCore pid=19610) WARNING 05-24 00:57:10 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|                         | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output:…

(EngineCore pid=19610) /home/jaspe/CSE151B/.venv-gpu/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=19610)   torch._check_is_size(blocksize)


Inference generation processing complete.


In [6]:
from tqdm import tqdm

data = [json.loads(line) for line in open(DATA_PATH)]

if 'SAMPLE_LIMIT' in globals() and SAMPLE_LIMIT is not None:
    start = SAMPLE_IDX*SAMPLE_LIMIT
    end = (SAMPLE_IDX+1)*SAMPLE_LIMIT
    data = data[start:end]

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
# Assuming SAVE_EVAL is defined in your config/environment
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    
    # Only try to fetch gold/answer if we are in training/dev mode
    correct = False
    gold = None
    
    if SAVE_EVAL:
        gold = item.get("answer")
        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                # Still keep the timeout for stability
                signal.alarm(3)
                correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
                signal.alarm(0)
            except Exception:
                signal.alarm(0)
                correct = False

    # Build the record dynamically based on availability
    record = {
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "response": response
    }

    print(item.get("id"))
    
    if SAVE_EVAL:
        record.update({"gold": gold, "correct": correct})
        
    results.append(record)

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 10843.60it/s]

0
1
2
3
4
Scoring complete. 5 results.


In [7]:
# mcq_res  = [r for r in results if r["is_mcq"]]
# free_res = [r for r in results if not r["is_mcq"]]

# def acc(subset):
#     return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

# print("=" * 50)
# print("EVALUATION RESULTS")
# print("=" * 50)
# print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
# print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
# print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
# print("=" * 50)

In [8]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)



In [9]:
def update_jsonl_by_id(out_path, new_results):
    # 1. Load existing data if it exists
    data_by_id = {}
    if os.path.exists(out_path):
        with open(out_path, "r") as f:
            for line in f:
                record = json.loads(line)
                data_by_id[record["id"]] = record
    
    # 2. Update/Add new results
    for r in new_results:
        # Determine schema based on your current SAVE_EVAL status
        if SAVE_EVAL:
            record = {
                "id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                "response": r["response"], "correct": r["correct"]
            }
        else:
            record = {
                "id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]
            }
        data_by_id[r["id"]] = record
        
    # 3. Rewrite the full file
    with open(out_path, "w") as f:
        for record in data_by_id.values():
            f.write(json.dumps(record) + "\n")
    print(f"Saved {len(results)} records to {out_path}")

# Use this function instead of your previous write loop
update_jsonl_by_id(out_path, results)

Saved 5 records to results/vllm_predictions.jsonl
